# 2B core benchmark

Complete comparison of `no_loc`, integer-coordinate `loc_text` and L40 `loc_embed` full runs, including matched shuffled-coordinate controls.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

evaluation_root = repo_root / "outputs" / "evaluation"
runs = {
    "no_loc": "11437",
    "loc_text integer": "11441",
    "loc_embed L40": "11438",
}
shuffled_runs = {
    "loc_text integer": "11445",
    "loc_embed L40": "11446",
}
condition_order = list(runs)

def read_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

summaries = {
    condition: read_json(evaluation_root / job / "scored_predictions" / "summary.json")
    for condition, job in runs.items()
}

predictions = {
    condition: pd.read_json(evaluation_root / job / "predictions.jsonl", lines=True)
    for condition, job in runs.items()
}

sample_scores = {
    condition: pd.read_json(
        evaluation_root / job / "scored_predictions" / "sample_scores.jsonl",
        lines=True,
    )
    for condition, job in runs.items()
}

pd.DataFrame({
    "Condition": condition_order,
    "Evaluation job": [runs[c] for c in condition_order],
    "Samples": [len(predictions[c]) for c in condition_order],
})

,Condition,Evaluation job,Samples
0,no_loc,11437,15029
1,loc_text integer,11441,15029
2,loc_embed L40,11438,15029


## Shuffled-coordinate controls

Values are shuffled minus correct. Negative values mean that replacing the true coordinates hurt performance.

In [2]:
shuffled_summaries = {
    condition: read_json(evaluation_root / job / "scored_predictions" / "summary.json")
    for condition, job in shuffled_runs.items()
}

def counterfactual_task_row(summary, task_type):
    return next(row for row in summary["by_task_type"] if row["task_type"] == task_type)

def primary_metrics(summary):
    return {
        "Caption BLEU-4": summary["captioning"]["bleu4"],
        "Binary accuracy": counterfactual_task_row(summary, "binary")["accuracy"],
        "MCQ accuracy": counterfactual_task_row(summary, "mcq")["accuracy"],
        "Bounding-box mIoU": counterfactual_task_row(summary, "bounding box")["miou"],
    }

counterfactual_rows = []
for condition in shuffled_runs:
    correct = primary_metrics(summaries[condition])
    shuffled = primary_metrics(shuffled_summaries[condition])
    counterfactual_rows.append({
        "Condition": condition,
        **{metric: shuffled[metric] - correct[metric] for metric in correct},
    })

counterfactual_deltas = pd.DataFrame(counterfactual_rows).set_index("Condition")
counterfactual_deltas.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-counterfactual_deltas.abs().to_numpy().max(), vmax=counterfactual_deltas.abs().to_numpy().max()).set_caption("Shuffled − correct coordinates")

,Caption BLEU-4,Binary accuracy,MCQ accuracy,Bounding-box mIoU
Condition,,,,
loc_text integer,-0.0713,-0.0039,-0.1180,-0.0009
loc_embed L40,-0.0739,-0.0097,-0.1351,+0.0009


### Direct-geography MCQs under shuffling

In [3]:
def category_accuracy(summary, category):
    return next(
        row["accuracy"]
        for row in summary["by_task_category"]
        if row["task_type"] == "mcq" and row["task_category"] == category
    )

geo_shuffle_rows = []
for condition in shuffled_runs:
    for category in ["country", "climate zone", "season"]:
        correct = category_accuracy(summaries[condition], category)
        shuffled = category_accuracy(shuffled_summaries[condition], category)
        geo_shuffle_rows.append({
            "Condition": condition,
            "Category": category,
            "Correct": correct,
            "Shuffled": shuffled,
            "Difference": shuffled - correct,
        })

pd.DataFrame(geo_shuffle_rows).style.format({"Correct": "{:.3f}", "Shuffled": "{:.3f}", "Difference": "{:+.3f}"})

,Condition,Category,Correct,Shuffled,Difference
0,loc_text integer,country,1.000,0.304,-0.696
1,loc_text integer,climate zone,0.984,0.483,-0.501
2,loc_text integer,season,0.956,0.821,-0.134
3,loc_embed L40,country,1.000,0.311,-0.689
4,loc_embed L40,climate zone,0.985,0.419,-0.566
5,loc_embed L40,season,0.970,0.809,-0.161


## Population check

In [4]:
id_sets = {condition: set(frame["sample_id"].astype(str)) for condition, frame in predictions.items()}
reference_ids = id_sets[condition_order[0]]

population_check = pd.DataFrame([
    {
        "Condition": condition,
        "Rows": len(predictions[condition]),
        "Unique sample IDs": predictions[condition]["sample_id"].astype(str).nunique(),
        "Same IDs as no_loc": ids == reference_ids,
    }
    for condition, ids in id_sets.items()
])
population_check

,Condition,Rows,Unique sample IDs,Same IDs as no_loc
0,no_loc,15029,15029,True
1,loc_text integer,15029,15029,True
2,loc_embed L40,15029,15029,True


## Main results

One primary metric per task family. Average rank weights the four task families equally; lower is better.

In [5]:
def task_row(summary, task_type):
    return next(row for row in summary["by_task_type"] if row["task_type"] == task_type)

main_results = pd.DataFrame([
    {
        "Condition": condition,
        "Caption BLEU-4": summary["captioning"]["bleu4"],
        "Binary accuracy": task_row(summary, "binary")["accuracy"],
        "MCQ accuracy": task_row(summary, "mcq")["accuracy"],
        "Bounding-box mIoU": task_row(summary, "bounding box")["miou"],
    }
    for condition, summary in summaries.items()
]).set_index("Condition").reindex(condition_order)

metric_columns = list(main_results.columns)
main_results["Average rank"] = main_results[metric_columns].rank(ascending=False).mean(axis=1)
main_results.style.format("{:.4f}").highlight_max(
    subset=metric_columns, axis=0, props="font-weight: bold"
).highlight_min(
    subset=["Average rank"], axis=0, props="font-weight: bold"
).set_caption("Primary benchmark metrics")

,Caption BLEU-4,Binary accuracy,MCQ accuracy,Bounding-box mIoU,Average rank
Condition,,,,,
no_loc,0.4414,0.7891,0.8020,0.5960,2.0000
loc_text integer,0.4424,0.7842,0.8038,0.5951,2.2500
loc_embed L40,0.4468,0.7871,0.8092,0.5872,1.7500


## Difference from no_loc

Positive values favor the location-conditioned model.

In [6]:
delta = main_results.loc[["loc_text integer", "loc_embed L40"], metric_columns].subtract(main_results.loc["no_loc", metric_columns])
limit = delta.abs().to_numpy().max()
delta.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-limit, vmax=limit).set_caption("Location condition − no_loc")

,Caption BLEU-4,Binary accuracy,MCQ accuracy,Bounding-box mIoU
Condition,,,,
loc_text integer,+0.0010,-0.0049,+0.0018,-0.0009
loc_embed L40,+0.0053,-0.0020,+0.0072,-0.0088


## Task-wise results

In [7]:
category_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_category"]:
        category_rows.append({"Condition": condition, **row})
category_scores = pd.DataFrame(category_rows)

def category_table(task_type, metric):
    rows = category_scores[category_scores["task_type"] == task_type]
    table = rows.pivot(index="Condition", columns="task_category", values=metric)
    overall = pd.Series({
        condition: task_row(summaries[condition], task_type)[metric]
        for condition in condition_order
    }, name="Overall")
    return table.reindex(condition_order).join(overall)

display(category_table("binary", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Binary accuracy"))
display(category_table("mcq", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("MCQ accuracy"))
display(category_table("bounding box", "miou").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Bounding-box mIoU"))

,adjacency,area,count,presence,Overall
Condition,,,,,
no_loc,0.741,0.887,0.800,0.785,0.789
loc_text integer,0.743,0.883,0.799,0.763,0.784
loc_embed L40,0.743,0.883,0.805,0.772,0.787


,adjacency,area,climate zone,count,country,presence,relative pos,season,Overall
Condition,,,,,,,,,
no_loc,0.699,0.721,0.969,0.586,0.974,0.842,0.804,0.956,0.802
loc_text integer,0.696,0.717,0.984,0.599,1.000,0.839,0.794,0.956,0.804
loc_embed L40,0.703,0.723,0.985,0.600,1.000,0.838,0.804,0.970,0.809


,point,reference,Overall
Condition,,,
no_loc,0.786,0.408,0.596
loc_text integer,0.786,0.405,0.595
loc_embed L40,0.791,0.385,0.587


## Geography-sensitive MCQs

Country, climate-zone and season questions are the clearest direct test of whether the location tokens carry useful geographic information.

In [8]:
geo_categories = ["country", "climate zone", "season"]
geo_mcq = (
    category_scores[
        (category_scores["task_type"] == "mcq")
        & category_scores["task_category"].isin(geo_categories)
    ]
    .pivot(index="Condition", columns="task_category", values="accuracy")
    .reindex(condition_order)
)
geo_mcq["Mean"] = geo_mcq.mean(axis=1)
geo_mcq.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Direct-geography MCQ accuracy")

task_category,climate zone,country,season,Mean
Condition,,,,
no_loc,0.969,0.974,0.956,0.966
loc_text integer,0.984,1.000,0.956,0.980
loc_embed L40,0.985,1.000,0.970,0.985


## Paired changes

Counts below show whether `loc_embed` fixes or breaks the exact same binary and MCQ samples.

In [9]:
base = sample_scores["no_loc"][["sample_id", "task_type", "task_category", "correct"]].rename(columns={"correct": "no_loc_correct"})
embed = sample_scores["loc_embed L40"][["sample_id", "correct"]].rename(columns={"correct": "loc_embed_correct"})
paired = base.merge(embed, on="sample_id", validate="one_to_one")
paired = paired[paired["task_type"].isin(["binary", "mcq"])].copy()

def transition(row):
    if row.no_loc_correct and not row.loc_embed_correct:
        return "Broken by loc_embed"
    if not row.no_loc_correct and row.loc_embed_correct:
        return "Fixed by loc_embed"
    if row.no_loc_correct:
        return "Both correct"
    return "Both wrong"

paired["Transition"] = paired.apply(transition, axis=1)
transition_table = pd.crosstab(
    [paired["task_type"], paired["task_category"]],
    paired["Transition"],
).fillna(0).astype(int)
transition_table

Transition               Both correct  Both wrong  Broken by loc_embed  \
task_type task_category                                                  
binary    adjacency              2039         655                   79   
          area                   1135         129                   24   
          count                  1020         230                   25   
          presence               1095         284                   49   
mcq       adjacency               859         331                   58   
          area                    449         159                   22   
          climate zone            649           5                    5   
          count                   312         212                    3   
          country                 304           0                    0   
          presence                580          98                   17   
          relative pos            489          93                   35   
          season                  664          16                    5   

Transition               Fixed by loc_embed  
task_type task_category                      
binary    adjacency                      84  
          area                           18  
          count                          31  
          presence                       30  
mcq       adjacency                      63  
          area                           23  
          climate zone                   16  
          count                          11  
          country                         8  
          presence                       14  
          relative pos                   35  
          season                         15

## Qualitative changes

Examples where the two models produced different answers. Change `task_filter` and `category_filter` to inspect a weakness or a location-sensitive subtask.

In [10]:
task_filter = "mcq"
category_filter = "country"
number_of_examples = 12

keep = ["sample_id", "input_text", "target_texts", "prediction", "task_type", "task_category", "country", "lat", "lon"]
base_predictions = predictions["no_loc"][keep].rename(columns={"prediction": "no_loc prediction"})
embed_predictions = predictions["loc_embed L40"][["sample_id", "prediction"]].rename(columns={"prediction": "loc_embed prediction"})
comparison = base_predictions.merge(embed_predictions, on="sample_id", validate="one_to_one")
changed = comparison[
    (comparison["task_type"] == task_filter)
    & (comparison["task_category"] == category_filter)
    & (comparison["no_loc prediction"] != comparison["loc_embed prediction"])
]
changed.head(number_of_examples)

,sample_id,input_text,target_texts,no_loc prediction,task_type,task_category,country,lat,lon,loc_embed prediction
2635,1093222,"From the following options, pick the one that ...",[d],b,mcq,country,Switzerland,47.242530,8.494720,d
2662,1097433,Which of the following countries is captured i...,[d],b,mcq,country,Switzerland,47.361504,8.541260,d
2675,1100631,Choose the country that is shown in the satell...,[c],d,mcq,country,Switzerland,47.404817,8.572694,c
2741,1109368,Choose one of the following options that repre...,[b],a,mcq,country,Switzerland,47.372693,8.652429,b
7080,4216731,Which country is shown in the satellite image?...,[c],d,mcq,country,Lithuania,54.422972,24.518370,c
9598,6502263,"From the following options, choose the country...",[d],a,mcq,country,Serbia,45.544342,18.983736,d
12667,8303471,Which country is shown in the satellite image?...,[a],c,mcq,country,Finland,60.115807,20.139283,a
13532,8734874,"From the following options, choose the country...",[d],b,mcq,country,Portugal,39.192915,-8.659781,d


## Full diagnostic tables

These retain sample counts, extraction rates, secondary caption metrics and bounding-box thresholds.

In [11]:
task_type_rows = []
caption_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_type"]:
        task_type_rows.append({"Condition": condition, **row})
    caption_rows.append({"Condition": condition, **summary["captioning"]})

display(pd.DataFrame(task_type_rows).sort_values(["task_type", "Condition"]).reset_index(drop=True))
display(pd.DataFrame(caption_rows).set_index("Condition").reindex(condition_order).reset_index())
display(category_scores.sort_values(["task_type", "task_category", "Condition"]).reset_index(drop=True))

,Condition,task_type,n,extraction_success,accuracy,correct,miou,acc@25,acc@50,acc@75,acc@90
0,loc_embed L40,binary,6927,1.0,0.787065,5452.0,NaN,NaN,NaN,NaN,NaN
1,loc_text integer,binary,6927,1.0,0.784178,5432.0,NaN,NaN,NaN,NaN,NaN
2,no_loc,binary,6927,1.0,0.789086,5466.0,NaN,NaN,NaN,NaN,NaN
3,loc_embed L40,bounding box,1582,1.0,NaN,NaN,0.587218,0.772440,0.673198,0.460809,0.180152
4,loc_text integer,bounding box,1582,1.0,NaN,NaN,0.595101,0.783186,0.675095,0.463338,0.189001
5,no_loc,bounding box,1582,1.0,NaN,NaN,0.596018,0.778761,0.685841,0.469659,0.185209
6,loc_embed L40,captioning,970,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,loc_text integer,captioning,970,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,no_loc,captioning,970,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,loc_embed L40,mcq,5550,1.0,0.809189,4491.0,NaN,NaN,NaN,NaN,NaN


,Condition,bleu1,bleu2,bleu3,bleu4,meteor,cider,rouge_1,rouge_2,rouge_l
0,no_loc,0.625581,0.537875,0.480875,0.441436,0.638279,1.544406,0.704022,0.550680,0.610700
1,loc_text integer,0.625035,0.537965,0.481506,0.442411,0.642687,1.554983,0.706997,0.553338,0.612295
2,loc_embed L40,0.627069,0.541094,0.485296,0.446766,0.640335,1.497573,0.705253,0.552546,0.612537


,Condition,task_type,task_category,n,extraction_success,accuracy,correct,miou,acc@25,acc@50,acc@75,acc@90
0,loc_embed L40,binary,adjacency,2857,1.0,0.743087,2123.0,NaN,NaN,NaN,NaN,NaN
1,loc_text integer,binary,adjacency,2857,1.0,0.743437,2124.0,NaN,NaN,NaN,NaN,NaN
2,no_loc,binary,adjacency,2857,1.0,0.741337,2118.0,NaN,NaN,NaN,NaN,NaN
3,loc_embed L40,binary,area,1306,1.0,0.882848,1153.0,NaN,NaN,NaN,NaN,NaN
4,loc_text integer,binary,area,1306,1.0,0.882848,1153.0,NaN,NaN,NaN,NaN,NaN
5,no_loc,binary,area,1306,1.0,0.887443,1159.0,NaN,NaN,NaN,NaN,NaN
6,loc_embed L40,binary,count,1306,1.0,0.804747,1051.0,NaN,NaN,NaN,NaN,NaN
7,loc_text integer,binary,count,1306,1.0,0.798622,1043.0,NaN,NaN,NaN,NaN,NaN
8,no_loc,binary,count,1306,1.0,0.800153,1045.0,NaN,NaN,NaN,NaN,NaN
9,loc_embed L40,binary,presence,1458,1.0,0.771605,1125.0,NaN,NaN,NaN,NaN,NaN


## Current reading

- `loc_embed` has the best MCQ accuracy; its gain is concentrated in direct-geography questions.
- `loc_text` has the best CIDEr, while `loc_embed` has the best BLEU-4, so captioning has no single winner across metrics.
- `no_loc` has the best binary accuracy and bounding-box mIoU.
- Shuffling strongly damages captioning and MCQ, confirming that both location-conditioned models use their coordinates.